# Lab 2: Logistic Regression

CS 300 - Introduction to Machine Learning

Date - Nov 24, 2026


## 1. Data Preparation and Exploration

In [25]:
import pandas as pd

data_filepath = "../datasets_globelEnv/ML_parami_1.csv"
df = pd.read_csv(data_filepath)
display(df.head())

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


In [26]:
print("\n1. Data Preparation and Exploration")

# a. Display the number of observations and variables.
print(f"Shape of df: {df.shape}")
print()

# b. Identify missing values in each variable.
print(df.isnull().sum())
print()

# c. Identify duplicated observations.
print(f"Number of duplicated rows: {df.duplicated().sum()}")
print()

# d. Examine the distribution of the target variable TenYearCHD.
print("Target variable counts: ")
print(df['TenYearCHD'].value_counts())
print()

# e. Comment on whether the target variable is balanced or imbalanced.
print("Target variable percentages: ")
print((df["TenYearCHD"].value_counts(normalize=True)*100).round(2))


1. Data Preparation and Exploration
Shape of df: (4238, 16)

male                 0
age                  0
education          105
currentSmoker        0
cigsPerDay          29
BPMeds              53
prevalentStroke      0
prevalentHyp         0
diabetes             0
totChol             50
sysBP                0
diaBP                0
BMI                 19
heartRate            1
glucose            388
TenYearCHD           0
dtype: int64

Number of duplicated rows: 0

Target variable counts: 
TenYearCHD
0    3594
1     644
Name: count, dtype: int64

Target variable percentages: 
TenYearCHD
0    84.8
1    15.2
Name: proportion, dtype: float64


a. Observations and Variables: The dataset contains 4,238 observations (rows) and 16 variables (columns). i.e. 15 input features and 1 binary target variable (TenYearCHD).

b. Missing Values: Seven variables have missing values: glucose (388), education (105), BPMeds (53), totChol (50), cigsPerDay (29), BMI (19) and heartRate (1). The remaining 9 columns have 0 missing values.

c. Duplicated Observations: There are 0 duplicated rows in the dataset.

d. Target Distribution (TenYearCHD):
|TenYearCHD|Total Patients|Percentage|
|---|---|---|
|0 (No CHD)| 3594 patients | 84.8% |
|1 (Yes CHD)| 644 patients | 15.2% |


e. Balanced or Imbalanced? The target variable is imbalanced with an approximate 85:15 ratio between negative (0) and positive (1) cases.

f. With imbalance dataset, predicting class 0 for everyone would give 84.80% accuracy on the full dataset while detecting zero CHD cases (fails to identify any actual heart disease patients). In medical diagnosis, missing sick patients (False Negative) is dangeous, so we must also look at other important metrics like Percision, Recall and F1-Score.

## 2. Train-Test Split 

In [27]:
from sklearn.model_selection import train_test_split

X = df.drop(columns='TenYearCHD')
y = df['TenYearCHD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

train_medians = X_train.median() # fill missing values with training set medians
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (2966, 15)
Testing set shape: (1272, 15)


The dataset is split into 70% training data (2,966 rows) and 30% testing data (1,272 rows). Setting random_state=42 ensure the split is reproducible every time and stratify=y preserves the target ratio in both sets. 

As the dataset contain NaN values, missing values are filled with the median of X_train.

## 3. Baseline Logistic Regression 

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)

# create function to avoid repetition
def evaluate_model(model, X_te, y_te, name = "Model Evaluation"):
    y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)
    cm = confusion_matrix(y_te, y_pred)

    print(f"--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)


base_model_lr = LogisticRegression(max_iter=5000, random_state=42)
base_model_lr.fit(X_train, y_train)

evaluate_model(base_model_lr, X_test, y_test, "Model A: Baseline Logistic Regression (Unscaled)")

'''
y_pred = base_model_lr.predict(X_test)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"--- Baseline Logistic Regression Score ---")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print("Confusion Matrix:\n", cm)'''


--- Model A: Baseline Logistic Regression (Unscaled) ---
Accuracy:  0.8467
Precision: 0.4643
Recall:    0.0674
F1-Score:  0.1176
Confusion Matrix:
 [[1064   15]
 [ 180   13]]


'\ny_pred = base_model_lr.predict(X_test)\nacc = accuracy_score(y_test, y_pred)\nprec = precision_score(y_test, y_pred)\nrec = recall_score(y_test, y_pred)\nf1 = f1_score(y_test, y_pred)\ncm = confusion_matrix(y_test, y_pred)\n\nprint(f"--- Baseline Logistic Regression Score ---")\nprint(f"Accuracy:  {acc:.4f}")\nprint(f"Precision: {prec:.4f}")\nprint(f"Recall:    {rec:.4f}")\nprint(f"F1-Score:  {f1:.4f}")\nprint("Confusion Matrix:\n", cm)'

Accuracy: The proportion of all predictions (both 0 and 1) that were correct.

Precision: Out of all patient the model predicted would develop CHD (1), how many actually did.

Recall: Out of all patients who actually developed CHD, how many the model successfully caught.

F1-Score: The balance between Precision and Recall metrics and it is useful for imbalanced dataset. Here very low score reflect the poor Recall.

Confusion Matrix: A table showing exact counts of True Negatives (TN), False Positives (FP), False Negatives (FN) and True Positives (TP).

## 4. Effect of Feature Scaling

In [29]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']
print("--- Numeric Columns ---")
print(numeric_cols)
print()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

scaled_model = LogisticRegression(max_iter=5000, random_state=42)
scaled_model.fit(X_train_scaled, y_train)

evaluate_model(base_model_lr, X_test, y_test, "Model A: Baseline Logistic Regression (Unscaled)")
print()
evaluate_model(scaled_model, X_test_scaled, y_test, "Model B: Scaled Logistic Regression")


--- Numeric Columns ---
['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose']

--- Model A: Baseline Logistic Regression (Unscaled) ---
Accuracy:  0.8467
Precision: 0.4643
Recall:    0.0674
F1-Score:  0.1176
Confusion Matrix:
 [[1064   15]
 [ 180   13]]

--- Model B: Scaled Logistic Regression ---
Accuracy:  0.8475
Precision: 0.4815
Recall:    0.0674
F1-Score:  0.1182
Confusion Matrix:
 [[1065   14]
 [ 180   13]]


Feature scaling produces only a very slight improvement in Accuracy, Precision and F1-Score. There has been some drop in Recall Score. 

## 5. Handling the Imbalanced Target Variable

In [30]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE

Random Undersampling: Randomly drops samples from the majority class until class counts are closer or equal. 

Random Oversampling: Randomly duplicates existing samples (with replacement) from the minority class to boost its count.

SMOTE (Synthetic Minority Over-sampling Technique): It balances imbalanced classification datasets by creating new, artificial data points for the underrepresented (minority) class instead of copying existing ones.

In [31]:
# Model A: Original training data (already trained as model_scaled)
evaluate_model(scaled_model, X_test_scaled, y_test, "Model A: Original Data (Scaled)")

# Model B: Random UnderSampling
rus = RandomOverSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train_scaled, y_train)

model_rus = LogisticRegression(max_iter=5000, random_state=42)
model_rus.fit(X_train_rus, y_train_rus)
evaluate_model(model_rus, X_test_scaled, y_test, "Model B: Random Undersampling (Scaled)")

# Model C: Random Oversampling
ros = RandomOverSampler(random_state=42)
X_train_ros, y_train_ros = ros.fit_resample(X_train_scaled, y_train)

model_ros = LogisticRegression(max_iter=5000, random_state=42)
model_ros.fit(X_train_ros, y_train_ros)
evaluate_model(model_ros, X_test_scaled, y_test, "Model C: Random Oversampling (Scaled)")

# Model D: SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

model_smote = LogisticRegression(max_iter=5000, random_state=42)
model_smote.fit(X_train_smote, y_train_smote)
evaluate_model(model_smote, X_test_scaled, y_test, "Model D: SMOTE (Scaled)")

--- Model A: Original Data (Scaled) ---
Accuracy:  0.8475
Precision: 0.4815
Recall:    0.0674
F1-Score:  0.1182
Confusion Matrix:
 [[1065   14]
 [ 180   13]]
--- Model B: Random Undersampling (Scaled) ---
Accuracy:  0.6635
Precision: 0.2516
Recall:    0.6166
F1-Score:  0.3574
Confusion Matrix:
 [[725 354]
 [ 74 119]]
--- Model C: Random Oversampling (Scaled) ---
Accuracy:  0.6635
Precision: 0.2516
Recall:    0.6166
F1-Score:  0.3574
Confusion Matrix:
 [[725 354]
 [ 74 119]]
--- Model D: SMOTE (Scaled) ---
Accuracy:  0.6549
Precision: 0.2349
Recall:    0.5648
F1-Score:  0.3318
Confusion Matrix:
 [[724 355]
 [ 84 109]]


## 6. Model Comparison

## 7. Final Conclusion